# Lab 16 — SVD: The Matrix Microscope

This lab is a full computational companion to Chapter 16.

The goal is not only to compute an SVD. The goal is to **see** what the SVD is doing.

We will explore:

1. matrices as transformations,
2. singular vectors as special input/output directions,
3. rank-one layers,
4. low-rank approximation,
5. image compression,
6. denoising,
7. least-squares instability,
8. PCA from SVD,
9. recommendation-style hidden factors,
10. high-dimensional low-rank structure.

The central formula is

$$
A = U\Sigma V^T.
$$

The central layer decomposition is

$$
A = \sum_{i=1}^r \sigma_i u_i v_i^T.
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

# Helper functions

def svd_rank_k(A, k):
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    return U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]

def energy_captured(s, k):
    return np.sum(s[:k]**2) / np.sum(s**2)

def show_matrix(M, title="Matrix", cmap="viridis"):
    plt.figure(figsize=(5, 4))
    plt.imshow(M, cmap=cmap, aspect="auto")
    plt.colorbar()
    plt.title(title)
    plt.xlabel("column")
    plt.ylabel("row")
    plt.show()


## 1. Warm-up: compute an SVD and reconstruct the matrix

NumPy returns the SVD in the form

```python
U, s, Vt = np.linalg.svd(A, full_matrices=False)
```

Here `s` is a one-dimensional array of singular values. The matrix $\Sigma$ is created with `np.diag(s)`.

The reconstruction is

```python
A_reconstructed = U @ np.diag(s) @ Vt
```

In [ ]:
A = np.array([[3, 1],
              [0, 2],
              [0, 0]], dtype=float)

U, s, Vt = np.linalg.svd(A, full_matrices=False)

print("A =")
print(A)
print("\nSingular values:")
print(s)
print("\nU =")
print(U)
print("\nVt =")
print(Vt)

A_rec = U @ np.diag(s) @ Vt
print("\nReconstructed A =")
print(A_rec)
print("\nReconstruction error:", np.linalg.norm(A - A_rec))


### Student task

Change the entries of `A`. Try a square matrix, a tall matrix, and a wide matrix. Observe the shapes of `U`, `s`, and `Vt` when `full_matrices=False`.

## 2. Geometric meaning: what does SVD do to the unit circle?

A $2 \times 2$ matrix maps the unit circle to an ellipse.

The right singular vectors point to the input directions that become the principal axes of that ellipse.
The singular values are the lengths of the semi-axes.
The left singular vectors point along the output axes.

In [ ]:
def plot_svd_geometry(A):
    theta = np.linspace(0, 2*np.pi, 400)
    circle = np.vstack([np.cos(theta), np.sin(theta)])
    image = A @ circle
    U, s, Vt = np.linalg.svd(A)
    V = Vt.T
    
    plt.figure(figsize=(7, 7))
    plt.plot(circle[0], circle[1], '--', label='unit circle')
    plt.plot(image[0], image[1], label='image under A')
    
    for i in range(2):
        v = V[:, i]
        u = U[:, i]
        plt.arrow(0, 0, v[0], v[1], head_width=0.05, length_includes_head=True)
        plt.text(1.08*v[0], 1.08*v[1], f'v{i+1}')
        plt.arrow(0, 0, s[i]*u[0], s[i]*u[1], head_width=0.05, length_includes_head=True)
        plt.text(1.04*s[i]*u[0], 1.04*s[i]*u[1], f'σ{i+1}u{i+1}')
    
    lim = max(3, np.max(np.abs(image))*1.2)
    plt.xlim(-lim, lim)
    plt.ylim(-lim, lim)
    plt.gca().set_aspect('equal')
    plt.grid(True)
    plt.legend()
    plt.title('SVD geometry: circle becomes ellipse')
    plt.show()

A2 = np.array([[2.0, 1.0],
               [0.5, 1.5]])
plot_svd_geometry(A2)
print("Singular values:", np.linalg.svd(A2, compute_uv=False))


### Student task

Try the following matrices:

```python
np.array([[3,0],[0,1]])
np.array([[1,2],[0,1]])
np.array([[0,-1],[1,0]])
np.array([[1,1],[1,1]])
```

Which ones stretch? Which ones rotate? Which ones collapse information?

## 3. Rank-one layers

The SVD layer decomposition is

$$
A = \sigma_1 u_1v_1^T + \sigma_2u_2v_2^T + \cdots + \sigma_r u_rv_r^T.
$$

Each layer is a rank-one matrix. Let us visualize these layers.

In [ ]:
A = np.array([[4, 4, 0, 0],
              [4, 4, 0, 0],
              [0, 0, 2, 2],
              [0, 0, 2, 2]], dtype=float)

U, s, Vt = np.linalg.svd(A, full_matrices=False)
print("Singular values:", s)
show_matrix(A, "Original matrix")

for i in range(4):
    layer = s[i] * np.outer(U[:, i], Vt[i, :])
    show_matrix(layer, f"Rank-one layer {i+1}, singular value {s[i]:.3f}")


### Interpretation

The first layer captures the strongest rectangular pattern. The second layer captures the strongest remaining pattern. If a singular value is essentially zero, its layer contributes essentially nothing.

## 4. Low-rank approximation and energy captured

The rank-$k$ approximation is

$$
A_k = \sum_{i=1}^k \sigma_i u_i v_i^T.
$$

The fraction of Frobenius energy captured is

$$
\frac{\sigma_1^2 + \cdots + \sigma_k^2}{\sigma_1^2 + \cdots + \sigma_r^2}.
$$

In [ ]:
np.random.seed(12)

# Construct a matrix with low-rank structure plus small noise
m, n = 40, 30
true_rank = 3
B = np.random.randn(m, true_rank) @ np.random.randn(true_rank, n)
A = B + 0.25*np.random.randn(m, n)

U, s, Vt = np.linalg.svd(A, full_matrices=False)

plt.figure(figsize=(6,4))
plt.plot(np.arange(1, len(s)+1), s, marker='o')
plt.xlabel('index')
plt.ylabel('singular value')
plt.title('Singular value decay')
plt.grid(True)
plt.show()

for k in [1, 2, 3, 5, 10, 20]:
    Ak = svd_rank_k(A, k)
    err = np.linalg.norm(A - Ak, 'fro')
    print(f"k={k:2d} | energy captured={energy_captured(s,k):6.2%} | Frobenius error={err:.3f}")


### Student task

Increase the noise level from `0.25` to `1.0` or `2.0`. What happens to the singular value decay? Is the low-rank structure still visible?

## 5. Image compression from scratch

We will create a synthetic grayscale image matrix. This avoids relying on external image files and makes the lab reproducible.

The image will contain:

- a smooth background,
- a circular bright region,
- stripe texture,
- and noise.

Then we will compress it using truncated SVD.

In [ ]:
# Synthetic image
N = 120
x = np.linspace(-1, 1, N)
y = np.linspace(-1, 1, N)
X, Y = np.meshgrid(x, y)

background = 0.35 + 0.25*X + 0.15*Y
circle = 0.45*((X+0.25)**2 + (Y-0.1)**2 < 0.22**2)
stripes = 0.10*np.sin(18*X + 6*Y)
noise = 0.06*np.random.randn(N, N)
img = np.clip(background + circle + stripes + noise, 0, 1)

show_matrix(img, "Synthetic image", cmap="gray")

U, s, Vt = np.linalg.svd(img, full_matrices=False)

plt.figure(figsize=(6,4))
plt.semilogy(np.arange(1, len(s)+1), s, marker='o', markersize=3)
plt.xlabel('index')
plt.ylabel('singular value (log scale)')
plt.title('Singular values of synthetic image')
plt.grid(True)
plt.show()


In [ ]:
for k in [1, 2, 5, 10, 20, 40]:
    approx = svd_rank_k(img, k)
    show_matrix(np.clip(approx, 0, 1), f"Rank-{k} approximation | energy={energy_captured(s,k):.1%}", cmap="gray")


### Storage calculation

For an $m \times n$ image, the original image stores $mn$ numbers.

A rank-$k$ SVD approximation stores approximately

$$
k(m+n+1)
$$

numbers.

This is useful when

$$
k(m+n+1) \ll mn.
$$

In [ ]:
m, n = img.shape
for k in [5, 10, 20, 40]:
    original = m*n
    compressed = k*(m+n+1)
    print(f"k={k:2d}: store {compressed:5d} numbers instead of {original:5d}; ratio={compressed/original:.2%}")


## 6. Denoising with SVD

If the meaningful signal is low-rank and the noise is spread across many directions, truncated SVD can remove some noise.

This is not magic. It works when the modeling assumption is reasonable.

In [ ]:
np.random.seed(0)

# Create a clean low-rank matrix
m, n, r = 80, 70, 4
clean = np.random.randn(m, r) @ np.random.randn(r, n)
noisy = clean + 1.2*np.random.randn(m, n)

U, s, Vt = np.linalg.svd(noisy, full_matrices=False)

for k in [2, 4, 8, 16]:
    denoised = svd_rank_k(noisy, k)
    error_to_clean = np.linalg.norm(clean - denoised, 'fro')
    error_noisy = np.linalg.norm(clean - noisy, 'fro')
    print(f"k={k:2d}: error to clean={error_to_clean:.2f}; noisy error={error_noisy:.2f}")


### Student task

Why might $k=4$ work well in this example? What would happen if the true rank were not known?

## 7. SVD and least-squares instability

Small singular values are dangerous in inverse problems because the pseudoinverse divides by them.

If $A$ has a tiny singular value $\sigma$, then the pseudoinverse contains $1/\sigma$, which can be huge.

In [ ]:
# A nearly singular matrix
A = np.array([[1.0, 1.0],
              [1.0, 1.0001]])

U, s, Vt = np.linalg.svd(A)
print("Singular values:", s)
print("Condition number:", s[0]/s[-1])

x_true = np.array([2.0, -1.0])
b = A @ x_true

for noise_level in [0, 1e-5, 1e-4, 1e-3]:
    b_noisy = b + noise_level*np.array([1.0, -1.0])
    x_est = np.linalg.pinv(A) @ b_noisy
    print(f"noise={noise_level:.0e}, solution={x_est}, error={np.linalg.norm(x_est-x_true):.3e}")


### Interpretation

The matrix nearly loses information in one direction. Small noise in the output can become a large error in the recovered input.

## 8. PCA from SVD

For a centered data matrix $X$, PCA is SVD:

$$
X = U\Sigma V^T.
$$

The columns of $V$ are principal directions. The coordinates of the data in the PCA system are $XV = U\Sigma$.

In [ ]:
np.random.seed(4)

# Two-dimensional correlated data
N = 300
z1 = np.random.randn(N)
z2 = 0.25*np.random.randn(N)
X = np.column_stack([3*z1 + z2, 1.5*z1 - z2])
X_centered = X - X.mean(axis=0)

U, s, Vt = np.linalg.svd(X_centered, full_matrices=False)
V = Vt.T
scores = X_centered @ V

plt.figure(figsize=(6,6))
plt.scatter(X_centered[:,0], X_centered[:,1], alpha=0.5)
for i in range(2):
    direction = V[:, i]
    length = s[i]/np.sqrt(N)
    plt.arrow(0, 0, length*direction[0], length*direction[1], head_width=0.12, length_includes_head=True)
    plt.text(1.05*length*direction[0], 1.05*length*direction[1], f'PC{i+1}')
plt.axis('equal')
plt.grid(True)
plt.title('PCA directions from SVD')
plt.show()

plt.figure(figsize=(6,4))
plt.scatter(scores[:,0], scores[:,1], alpha=0.5)
plt.axhline(0)
plt.axvline(0)
plt.grid(True)
plt.title('Data in principal-component coordinates')
plt.xlabel('PC1 score')
plt.ylabel('PC2 score')
plt.show()

print("Singular values:", s)
print("Energy captured by PC1:", energy_captured(s, 1))


## 9. Recommendation-style hidden factors

A low-rank matrix can model hidden factors.

Rows might be users. Columns might be movies. Entries might be ratings.

A rank-$k$ approximation says that a small number of hidden taste directions explain much of the rating pattern.

In [ ]:
np.random.seed(10)

users = 12
items = 9
hidden = 2

user_factors = np.random.randn(users, hidden)
item_factors = np.random.randn(hidden, items)
ratings = user_factors @ item_factors + 0.3*np.random.randn(users, items)

U, s, Vt = np.linalg.svd(ratings, full_matrices=False)
print("Singular values:", np.round(s, 3))

show_matrix(ratings, "Synthetic user-item rating matrix")

for k in [1, 2, 4]:
    Rk = svd_rank_k(ratings, k)
    show_matrix(Rk, f"Rank-{k} approximation | energy={energy_captured(s,k):.1%}")


### Student task

What might the first two hidden factors represent? In real recommendation systems, why do we need more than plain SVD?

## 10. High-dimensional low-rank structure

Modern datasets are often high-dimensional. SVD is useful because many high-dimensional datasets have approximate low-rank structure.

For example, a dataset may have $500$ features, but most variation may live near a $5$-dimensional subspace.

In [ ]:
np.random.seed(123)

n_samples = 400
n_features = 200
latent_dim = 5

Z = np.random.randn(n_samples, latent_dim)
W = np.random.randn(latent_dim, n_features)
X = Z @ W + 0.5*np.random.randn(n_samples, n_features)
X = X - X.mean(axis=0)

U, s, Vt = np.linalg.svd(X, full_matrices=False)

plt.figure(figsize=(7,4))
plt.plot(np.arange(1, 51), s[:50], marker='o')
plt.xlabel('component')
plt.ylabel('singular value')
plt.title('Singular values of high-dimensional data')
plt.grid(True)
plt.show()

cumulative = np.cumsum(s**2)/np.sum(s**2)
plt.figure(figsize=(7,4))
plt.plot(np.arange(1, 51), cumulative[:50], marker='o')
plt.xlabel('number of components')
plt.ylabel('cumulative energy')
plt.title('Energy captured by top SVD components')
plt.grid(True)
plt.show()

for k in [2, 5, 10, 20]:
    print(f"k={k:2d}: energy captured={energy_captured(s,k):.2%}")


## Final reflection

Answer these questions in your own words.

1. What does SVD reveal that is not obvious from looking at matrix entries?
2. Why are rank-one layers useful for understanding images and data tables?
3. Why do singular values help us decide how many layers to keep?
4. What is the connection between SVD and PCA?
5. What can go wrong if we use a low-rank approximation too aggressively?

## Mini-project options

Choose one.

### Option A: Image compression

Create your own synthetic image matrix and compare rank-$k$ approximations for several values of $k$.

### Option B: Denoising

Create a low-rank matrix, add noise, and test which value of $k$ gives the best reconstruction.

### Option C: Recommendation factors

Create a user-item matrix from hidden factors and use SVD to recover approximate structure.

### Option D: High-dimensional data

Create data with a small latent dimension inside a high-dimensional feature space. Use singular values to detect the latent dimension.